在Agent运行中访问长期记忆

1 在工具、中间件 中访问-基于InMemoryStore实现

In [4]:
import os

from langgraph.store.postgres import PostgresStore

from tool import response

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union, NotRequired
from typing import  Any,Callable
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich import print as rprint

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.messages import RemoveMessage
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model

from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.prebuilt import ToolRuntime
from langgraph.types import Command
from langchain_core.tools import tool

load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myStore=InMemoryStore()


#2自定义1个继承于AgentState的类
class CustomState(AgentState):
    user_id:NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime)->str:
    """
    将用户信息保存在长期记忆中

    Args:
        name:用户名
        runtime:工具的运行时

    Returns:
        str:保存的状态
    """
    namespace=("users", )
    key=runtime.state["user_id"]
    value={"name":name}
    runtime.store.put(namespace,key,value)
    return  "Success"

@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime)->str:
    """
    从长期记忆中读取客户信息

    Args:
        runtime:工具的运行时

    Returns:
        str:用户信息
    """
    namespace=("users", )
    key=runtime.state["user_id"]
    item=runtime.store.get(namespace,key)
    return str(item.value) if item else "Unknow"
#4初始化1个agent实例
myAgent=create_agent(
    model=myModel,
    name="琳琳",
    store=myStore,
    tools=[get_user_info,save_user_info],
    state_schema=CustomState,
    system_prompt="用户提及个人信息的时候，可以使用注册的工具save_user_info来保存用户信息 用户询问个人信息的时候，可以使用注册的工具get_user_info来查询用户信息"
)

#4 创建1个config文件，保证同1个thread id是共享记忆的
# myConfig={
#     "configurable":{
#         "thread_id":"1"
#     }
# }

response1=myAgent.invoke({"messages": [HumanMessage("你好我是小向")],"user_id":"user-1"},)
for msg in response1["messages"]:
        msg.pretty_print()

response2=myAgent.invoke({"messages": [ HumanMessage("我是男生，身高175，体重82kg，最近在健身，然后失业了，在学习langchin框架，fastapi，pyPDF,准备找ai大模型应用相关的岗位")],"user_id":"user-1"})
for msg in response2["messages"]:
        msg.pretty_print()
response3=myAgent.invoke({"messages": [ HumanMessage("请问我是谁，身高体重多少")],"user_id":"user-1"})
for msg in response3["messages"]:
        msg.pretty_print()

================================ Human Message =================================

你好我是小向
================================== Ai Message ==================================
Name: 琳琳

你好小向！很高兴认识你 😊 让我先把你的信息记录下来。
Tool Calls:
  save_user_info (call_b21096074c574dabbf79362c)
 Call ID: call_b21096074c574dabbf79362c
  Args:
    name: 小向
================================= Tool Message =================================
Name: save_user_info

Success
================================== Ai Message ==================================
Name: 琳琳

已经帮你保存好啦，小向！✅

有什么我可以帮你的吗？比如：
- 查询你保存的个人信息
- 解答问题、协助处理各种任务

随时告诉我～
================================ Human Message =================================

我是男生，身高175，体重82kg，最近在健身，然后失业了，在学习langchin框架，fastapi，pyPDF,准备找ai大模型应用相关的岗位
================================== Ai Message ==================================
Name: 琳琳
Tool Calls:
  save_user_info (call_4488cd09d1af442ea4f441da)
 Call ID: call_4488cd09d1af442ea4f441da
  Args:
    name: 男生｜身高175cm｜体重82kg｜正在健身；目前失业，学习LangCh

基于PostgreSQL的实现

In [8]:
import os
from tool import response

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union, NotRequired
from typing import  Any,Callable
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich import print as rprint

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.messages import RemoveMessage
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model

from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import PostgresStore
from langgraph.prebuilt import ToolRuntime
from langgraph.types import Command
from langchain_core.tools import tool

load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)




#2自定义1个继承于AgentState的类
class CustomState(AgentState):
    user_id:NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime)->str:
    """
    将用户信息保存在长期记忆中

    Args:
        name:用户名
        runtime:工具的运行时

    Returns:
        str:保存的状态
    """
    namespace=("users", )
    key=runtime.state["user_id"]
    value={"name":name}
    runtime.store.put(namespace,key,value)
    return  "Success"

@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime)->str:
    """
    从长期记忆中读取客户信息

    Args:
        runtime:工具的运行时

    Returns:
        str:用户信息
    """
    namespace=("users", )
    key=runtime.state["user_id"]
    item=runtime.store.get(namespace,key)
    return str(item.value) if item else "Unknow"
DB_URL=os.getenv("DB_URL")
with PostgresStore.from_conn_string(DB_URL) as myStore:
    myStore.setup()
    #4初始化1个agent实例
    myAgent=create_agent(
        model=myModel,
        name="琳琳",
        store=myStore,
        tools=[get_user_info,save_user_info],
        state_schema=CustomState,
        system_prompt="用户提及个人信息的时候，可以使用注册的工具save_user_info来保存用户信息 用户询问个人信息的时候，可以使用注册的工具get_user_info来查询用户信息"
    )

    #4 创建1个config文件，保证同1个thread id是共享记忆的
    # myConfig={
    #     "configurable":{
    #         "thread_id":"1"
    #     }
    # }

    response1=myAgent.invoke({"messages": [HumanMessage("你好我叫Aureliano")],"user_id":"user-1"},)
    for msg in response1["messages"]:
            msg.pretty_print()

    response2=myAgent.invoke({"messages": [ HumanMessage("我是男生，身高175，体重82kg，最近在健身，然后失业了，在学习langchin框架，fastapi，pyPDF,准备找ai大模型应用相关的岗位")],"user_id":"user-1"})
    for msg in response2["messages"]:
            msg.pretty_print()
    response3=myAgent.invoke({"messages": [ HumanMessage("请问我是谁，身高体重多少")],"user_id":"user-1"})
    for msg in response3["messages"]:
            msg.pretty_print()
    response4=myAgent.invoke({"messages": [ HumanMessage("请问我叫什么")],"user_id":"user-1"})
    for msg in response4["messages"]:
            msg.pretty_print()

================================ Human Message =================================

你好我叫Aureliano
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你！让我把你的信息保存一下。
Tool Calls:
  save_user_info (call_995cbc0c8c734d9ea9f7f902)
 Call ID: call_995cbc0c8c734d9ea9f7f902
  Args:
    name: Aureliano
================================= Tool Message =================================
Name: save_user_info

Success
================================== Ai Message ==================================
Name: 琳琳

你好，Aureliano！很高兴认识你！😊

我已经把你的名字保存好了。如果你需要查询或更新个人信息，随时告诉我哦！有什么我可以帮你的吗？
================================ Human Message =================================

我是男生，身高175，体重82kg，最近在健身，然后失业了，在学习langchin框架，fastapi，pyPDF,准备找ai大模型应用相关的岗位
================================== Ai Message ==================================
Name: 琳琳

我先帮你把这些信息记录到长期记忆里，稍等一下～
Tool Calls:
  get_user_info (call_3050bd7766c642e095583e01)
 Call ID: call_3050bd7766c642e095583e01
  Args:
